## Data generation 

This notebook generates bakery data based on databricks samples.bakehouse dataset 

In [0]:
dbutils.widgets.text("catalog", "workspace", "Catalog name")
dbutils.widgets.text("schema", "lakeflow_examples", "Schema name")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

print(f"Catalog: {catalog}")
print(f"Schema: {schema}")

#### Components

Classes for generating the data.

- DataGenerator: will generate data based on a given Delta table. The generated data will have the same schema as the source table
- DatasourceConfig: configuration for the source table
- GeneratorConfig: configuration for the DataGenerator

In [0]:
from dataclasses import dataclass
import csv
import time

@dataclass
class DatasourceConfig:
    """Class for storing generator's configuration."""
    source_catalog: str
    source_schema: str
    source_table: str

    def table_path(self) -> str:
        return f"{self.source_catalog}.{self.source_schema}.{self.source_table}"

class DataGenerator:
    def __init__(self, conf: DatasourceConfig):
        self.conf = conf

    def generate(self, batch_size: int, target_file: str, batch_interval: float = 0.1):
        """Generate data for the given table in batches and write to the target file.

        Args:
            batch_size (int): Number of rows to generate in each batch.
            target_file (str): Path to the target file.
            batch_interval (int, optional): Interval between batches in seconds. Defaults to 1.
        """

        if batch_size <= 0:
            raise ValueError("Batch size must be greater than 0.")

        if batch_interval <= 0:
            raise ValueError("Batch interval must be greater than 0.")

        if target_file == "":
            raise ValueError("Target file must be specified.")

        source_table = self.conf.table_path()
        source_df = spark.read.table(source_table)
        total_rows = source_df.count()

        n = 0
        if total_rows % batch_size == 0:
            n = total_rows // batch_size
        else:
            n = total_rows // batch_size + 1

        data = source_df.collect()
        columns = source_df.columns
        for i in range(n):
            batch = data[i * batch_size: (i + 1) * batch_size]
            self._write_batch(target_file + f"_{i}", batch, columns, batch_interval)
            time.sleep(batch_interval)

    def _write_batch(self, target_file: str, data: list, columns: list[str], batch_interval: int):
        if not data:
            raise ValueError("Data must be specified.")
        
        if columns is None:
            raise ValueError("Columns must be specified.")

        with open(target_file, 'w', newline='') as csvfile:
                writer = csv.DictWriter(csvfile, fieldnames=columns)

                writer.writeheader()
                for row in data:
                    writer.writerow(row.asDict())



In [0]:
dbutils.fs.ls("/Volumes/workspace/lakeflow_examples/bakery")

In [0]:
source_df = spark.read.table("samples.bakehouse.sales_transactions")

# rows = source_df.collect()
# rows[0].asDict()

conf = DatasourceConfig(catalog, schema, "sales_transactions")
DataGenerator(conf).generate(
    batch_size=100, 
    target_file= "/Volumes/workspace/lakeflow_examples/bakery/sales_transactions.csv", 
    batch_interval= 0.1)